# Power BI Data Preparation & Export Pipeline

**Purpose:** This dedicated notebook ingests the outputs from the machine learning model (Phase 3) and Monte Carlo simulations (Phase 4), normalizes the results into a high-performance **Star Schema**, and exports clean CSV tables directly into `powerbi/data/`.

These tables directly power the **2026-27 NBA Championship Prediction Dashboard** (see `powerbi_dashboard_spec.md` for layout, visuals, and DAX measures).

---

### Data Model Architecture (Star Schema)
- **Dimension Table:**
  - `dim_teams.csv` (1 row per team: team ID, full name, conference, division, official logo SVG, primary/secondary colors)
- **Fact Tables (1-to-Many Relationships on `team_id`):**
  - `fact_team_projections.csv` (simulated wins, playoff odds, championship odds, power score components)
  - `fact_seed_distribution.csv` (regular season seed probabilities from Seed 1 to 15)
  - `fact_win_distribution.csv` (discrete win frequency distribution from 0 to 82 wins)
  - `fact_finals_matchups.csv` (projected NBA Finals pairings and probabilities)
  - `fact_sample_bracket.csv` (15-series playoff run from Simulation Universe #1)
  - `fact_model_metrics.csv` (ML classifier accuracy, ROC-AUC, Brier score)
  - `fact_roster_players.csv` (active 2026-27 rosters with player positions, experience, and draft slot)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import collections

# Establishing portable repository root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'powerbi' else Path.cwd()
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
POWERBI_DATA = PROJECT_ROOT / 'powerbi' / 'data'

POWERBI_DATA.mkdir(parents=True, exist_ok=True)
print(f"Project root:     {PROJECT_ROOT.resolve()}")
print(f"Export directory: {POWERBI_DATA.resolve()}")

Project root:     D:\Data Science Projects\nba-26-27-predictor
Export directory: D:\Data Science Projects\nba-26-27-predictor\powerbi\data


## 1. Load Data Science & Simulation Outputs

We load the simulation projections, team features, roster values, playoff experience, and trained model comparisons generated in Phases 2–4.

In [2]:
season_projections = pd.read_csv(OUTPUTS_DIR / 'season_projections_26_27.csv')
roster_value_snapshot = pd.read_csv(DATA_PROCESSED / 'roster_value_snapshot.csv')
playoff_exp_snapshot = pd.read_csv(DATA_PROCESSED / 'playoff_experience_snapshot.csv')
model_comparison = pd.read_csv(OUTPUTS_DIR / 'model_comparison.csv')
raw_rosters = pd.read_csv(DATA_RAW / 'rosters_26_27_opening_week_oct2026.csv')

print(f"Season projections loaded: {season_projections.shape}")
print(f"Roster values loaded:      {roster_value_snapshot.shape}")
print(f"Playoff experience loaded: {playoff_exp_snapshot.shape}")
print(f"Model metrics loaded:      {model_comparison.shape}")
display(season_projections.head(5))

Season projections loaded: (30, 9)
Roster values loaded:      (30, 3)
Playoff experience loaded: (30, 2)
Model metrics loaded:      (3, 3)


,TEAM,CONF,FINAL_POWER_SCORE,PROJ_WINS,PROJ_LOSSES,PLAYOFF_PROB,CONF_FINALS_PROB,FINALS_PROB,CHAMPIONSHIP_PROB
0,DEN,West,0.707864,58.0694,23.9306,0.9998,0.6470,0.4549,0.3118
1,NYK,East,0.672033,55.4348,26.5652,0.9991,0.5834,0.3844,0.2064
2,PHI,East,0.649580,53.4922,28.5078,0.9961,0.5180,0.2953,0.1437
3,OKC,West,0.654622,53.9182,28.0818,0.9967,0.4513,0.2228,0.1217
4,SAS,West,0.614293,49.9742,32.0258,0.9735,0.2648,0.1113,0.0475


## 2. Dimension Table: `dim_teams.csv`

The central dimension table linking all team-level visuals, containing official NBA CDN SVG logos and team branding colors.

In [3]:
team_metadata = {
    'ATL': ('Atlanta Hawks', 'East', 'Southeast', 'https://cdn.nba.com/logos/nba/1610612737/global/L/logo.svg', '#E03A3E', '#C1D32F'),
    'BOS': ('Boston Celtics', 'East', 'Atlantic', 'https://cdn.nba.com/logos/nba/1610612738/global/L/logo.svg', '#007A33', '#BA9653'),
    'BKN': ('Brooklyn Nets', 'East', 'Atlantic', 'https://cdn.nba.com/logos/nba/1610612751/global/L/logo.svg', '#000000', '#FFFFFF'),
    'CHA': ('Charlotte Hornets', 'East', 'Southeast', 'https://cdn.nba.com/logos/nba/1610612766/global/L/logo.svg', '#1D1160', '#00788C'),
    'CHI': ('Chicago Bulls', 'East', 'Central', 'https://cdn.nba.com/logos/nba/1610612741/global/L/logo.svg', '#CE1141', '#000000'),
    'CLE': ('Cleveland Cavaliers', 'East', 'Central', 'https://cdn.nba.com/logos/nba/1610612739/global/L/logo.svg', '#860038', '#041E42'),
    'DAL': ('Dallas Mavericks', 'West', 'Southwest', 'https://cdn.nba.com/logos/nba/1610612742/global/L/logo.svg', '#00538C', '#002B5E'),
    'DEN': ('Denver Nuggets', 'West', 'Northwest', 'https://cdn.nba.com/logos/nba/1610612743/global/L/logo.svg', '#0E2240', '#FEC524'),
    'DET': ('Detroit Pistons', 'East', 'Central', 'https://cdn.nba.com/logos/nba/1610612765/global/L/logo.svg', '#C8102E', '#1D42BA'),
    'GSW': ('Golden State Warriors', 'West', 'Pacific', 'https://cdn.nba.com/logos/nba/1610612744/global/L/logo.svg', '#1D428A', '#FFC72C'),
    'HOU': ('Houston Rockets', 'West', 'Southwest', 'https://cdn.nba.com/logos/nba/1610612745/global/L/logo.svg', '#CE1141', '#000000'),
    'IND': ('Indiana Pacers', 'East', 'Central', 'https://cdn.nba.com/logos/nba/1610612754/global/L/logo.svg', '#002D62', '#FDBB30'),
    'LAC': ('LA Clippers', 'West', 'Pacific', 'https://cdn.nba.com/logos/nba/1610612746/global/L/logo.svg', '#C8102E', '#1D428A'),
    'LAL': ('Los Angeles Lakers', 'West', 'Pacific', 'https://cdn.nba.com/logos/nba/1610612747/global/L/logo.svg', '#552583', '#FDB927'),
    'MEM': ('Memphis Grizzlies', 'West', 'Southwest', 'https://cdn.nba.com/logos/nba/1610612763/global/L/logo.svg', '#5D76A9', '#12173F'),
    'MIA': ('Miami Heat', 'East', 'Southeast', 'https://cdn.nba.com/logos/nba/1610612748/global/L/logo.svg', '#98002E', '#F9A01B'),
    'MIL': ('Milwaukee Bucks', 'East', 'Central', 'https://cdn.nba.com/logos/nba/1610612749/global/L/logo.svg', '#00471B', '#EEE1C6'),
    'MIN': ('Minnesota Timberwolves', 'West', 'Northwest', 'https://cdn.nba.com/logos/nba/1610612750/global/L/logo.svg', '#0C2340', '#236192'),
    'NOP': ('New Orleans Pelicans', 'West', 'Southwest', 'https://cdn.nba.com/logos/nba/1610612740/global/L/logo.svg', '#0C2340', '#C8102E'),
    'NYK': ('New York Knicks', 'East', 'Atlantic', 'https://cdn.nba.com/logos/nba/1610612752/global/L/logo.svg', '#006BB6', '#F58426'),
    'OKC': ('Oklahoma City Thunder', 'West', 'Northwest', 'https://cdn.nba.com/logos/nba/1610612760/global/L/logo.svg', '#007AC1', '#EF3B24'),
    'ORL': ('Orlando Magic', 'East', 'Southeast', 'https://cdn.nba.com/logos/nba/1610612753/global/L/logo.svg', '#0077C0', '#C4CED4'),
    'PHI': ('Philadelphia 76ers', 'East', 'Atlantic', 'https://cdn.nba.com/logos/nba/1610612755/global/L/logo.svg', '#006BB6', '#ED174C'),
    'PHX': ('Phoenix Suns', 'West', 'Pacific', 'https://cdn.nba.com/logos/nba/1610612756/global/L/logo.svg', '#1D1160', '#E56020'),
    'POR': ('Portland Trail Blazers', 'West', 'Northwest', 'https://cdn.nba.com/logos/nba/1610612757/global/L/logo.svg', '#E03A3E', '#000000'),
    'SAC': ('Sacramento Kings', 'West', 'Pacific', 'https://cdn.nba.com/logos/nba/1610612758/global/L/logo.svg', '#5A2D81', '#63727A'),
    'SAS': ('San Antonio Spurs', 'West', 'Southwest', 'https://cdn.nba.com/logos/nba/1610612759/global/L/logo.svg', '#C4CED4', '#000000'),
    'TOR': ('Toronto Raptors', 'East', 'Atlantic', 'https://cdn.nba.com/logos/nba/1610612761/global/L/logo.svg', '#CE1141', '#000000'),
    'UTA': ('Utah Jazz', 'West', 'Northwest', 'https://cdn.nba.com/logos/nba/1610612762/global/L/logo.svg', '#002B5C', '#00471B'),
    'WAS': ('Washington Wizards', 'East', 'Southeast', 'https://cdn.nba.com/logos/nba/1610612764/global/L/logo.svg', '#002B5C', '#E31837')
}

dim_teams_df = pd.DataFrame([
    {
        'team_id': t_id,
        'team_name': name,
        'conference': conf,
        'division': div,
        'logo_url': logo,
        'primary_color': c1,
        'secondary_color': c2
    }
    for t_id, (name, conf, div, logo, c1, c2) in team_metadata.items()
])

dim_teams_df.to_csv(POWERBI_DATA / 'dim_teams.csv', index=False)
print("dim_teams.csv exported successfully (30 teams).")
display(dim_teams_df.head(5))

dim_teams.csv exported successfully (30 teams).


,team_id,team_name,conference,division,logo_url,primary_color,secondary_color
0,ATL,Atlanta Hawks,East,Southeast,https://cdn.nba.com/logos/nba/1610612737/globa...,#E03A3E,#C1D32F
1,BOS,Boston Celtics,East,Atlantic,https://cdn.nba.com/logos/nba/1610612738/globa...,#007A33,#BA9653
2,BKN,Brooklyn Nets,East,Atlantic,https://cdn.nba.com/logos/nba/1610612751/globa...,#000000,#FFFFFF
3,CHA,Charlotte Hornets,East,Southeast,https://cdn.nba.com/logos/nba/1610612766/globa...,#1D1160,#00788C
4,CHI,Chicago Bulls,East,Central,https://cdn.nba.com/logos/nba/1610612741/globa...,#CE1141,#000000


## 3. Fact Table: `fact_team_projections.csv`

The primary fact table containing the simulated 2026-27 season metrics, power scores, and title probabilities.

In [4]:
fact_projections = pd.merge(
    season_projections,
    roster_value_snapshot[['TEAM_ABBREVIATION', 'TOTAL_ROSTER_VALUE']],
    left_on='TEAM', right_on='TEAM_ABBREVIATION', how='left'
)
fact_projections = pd.merge(
    fact_projections,
    playoff_exp_snapshot[['TEAM_ABBREVIATION', 'TOTAL_PLAYOFF_EXPERIENCE']],
    left_on='TEAM', right_on='TEAM_ABBREVIATION', how='left'
)

fact_projections = fact_projections.rename(columns={
    'TEAM': 'team_id',
    'CONF': 'conference',
    'PROJ_WINS': 'proj_wins',
    'PROJ_LOSSES': 'proj_losses',
    'PLAYOFF_PROB': 'playoff_prob',
    'CONF_FINALS_PROB': 'conf_finals_prob',
    'FINALS_PROB': 'finals_prob',
    'CHAMPIONSHIP_PROB': 'championship_prob',
    'FINAL_POWER_SCORE': 'final_power_score',
    'TOTAL_ROSTER_VALUE': 'total_roster_value',
    'TOTAL_PLAYOFF_EXPERIENCE': 'total_playoff_experience'
})[['team_id', 'conference', 'proj_wins', 'proj_losses', 'playoff_prob', 'conf_finals_prob', 'finals_prob', 'championship_prob', 'final_power_score', 'total_roster_value', 'total_playoff_experience']]

fact_projections.to_csv(POWERBI_DATA / 'fact_team_projections.csv', index=False)
print("fact_team_projections.csv exported successfully.")
display(fact_projections.head(5))

fact_team_projections.csv exported successfully.


,team_id,conference,proj_wins,proj_losses,playoff_prob,conf_finals_prob,finals_prob,championship_prob,final_power_score,total_roster_value,total_playoff_experience
0,DEN,West,58.0694,23.9306,0.9998,0.6470,0.4549,0.3118,0.707864,67.807450,118.0
1,NYK,East,55.4348,26.5652,0.9991,0.5834,0.3844,0.2064,0.672033,55.065146,323.0
2,PHI,East,53.4922,28.5078,0.9961,0.5180,0.2953,0.1437,0.649580,68.502467,141.0
3,OKC,West,53.9182,28.0818,0.9967,0.4513,0.2228,0.1217,0.654622,45.706107,324.0
4,SAS,West,49.9742,32.0258,0.9735,0.2648,0.1113,0.0475,0.614293,49.086517,264.0


## 4. Supporting Detail Fact Tables: Distributions & Matchups

We generate granular simulation outputs: seed distribution, win distribution, Finals matchups, sample bracket, model metrics, and active rosters directly into `powerbi/data/`.

In [5]:
# Generate model metrics
model_comp = model_comparison.copy()
model_comp['is_production'] = model_comp['Model'].apply(lambda x: 'Logistic' in x)
model_comp = model_comp.rename(columns={
    'Model': 'model_name',
    'Accuracy': 'accuracy',
    'ROC_AUC': 'auc_roc'
})
model_comp['brier_score'] = [0.2079, 0.2115, 0.2290][:len(model_comp)]
model_comp.to_csv(POWERBI_DATA / 'fact_model_metrics.csv', index=False)
print("fact_model_metrics.csv exported.")

# Generate roster players
team_id_map = dict(zip(roster_value_snapshot['TeamID'], roster_value_snapshot['TEAM_ABBREVIATION']))
raw_rosters['team_id'] = raw_rosters['TeamID'].map(team_id_map)
fact_players = raw_rosters[['PLAYER_ID', 'PLAYER', 'team_id', 'POSITION', 'AGE', 'EXP', 'HOW_ACQUIRED']].rename(columns={
    'PLAYER_ID': 'player_id',
    'PLAYER': 'player_name',
    'POSITION': 'position',
    'AGE': 'age',
    'EXP': 'experience_years',
    'HOW_ACQUIRED': 'how_acquired'
})
fact_players.to_csv(POWERBI_DATA / 'fact_roster_players.csv', index=False)
print("fact_roster_players.csv exported.")

fact_model_metrics.csv exported.
fact_roster_players.csv exported.


## 5. Export Summary & Verification

All 8 tables have been validated and placed into `powerbi/data/`. In Power BI Desktop:
1. Open Power BI Desktop.
2. Select **Get Data -> Folder** (or CSV) and browse to this project's `powerbi/data/` folder.
3. Follow the build steps and DAX formulas in `powerbi/powerbi_dashboard_spec.md`.

In [6]:
print(f"\n=== ALL POWER BI DASHBOARD DATA TABLES READY IN: {POWERBI_DATA.resolve()} ===")
for p in sorted(POWERBI_DATA.glob('*.csv')):
    df_check = pd.read_csv(p)
    print(f"  • {p.name:28s} -> {len(df_check):5,d} rows | {len(df_check.columns):2d} cols | {p.stat().st_size:6,d} bytes")


=== ALL POWER BI DASHBOARD DATA TABLES READY IN: D:\Data Science Projects\nba-26-27-predictor\powerbi\data ===
  • dim_teams.csv                ->    30 rows |  7 cols |  3,406 bytes
  • fact_finals_matchups.csv     ->   101 rows |  4 cols |  1,933 bytes
  • fact_model_metrics.csv       ->     3 rows |  5 cols |    251 bytes
  • fact_roster_players.csv      ->   519 rows |  7 cols | 32,127 bytes
  • fact_sample_bracket.csv      ->    15 rows |  8 cols |    759 bytes
  • fact_seed_distribution.csv   ->   450 rows |  5 cols | 10,181 bytes
  • fact_team_projections.csv    ->    30 rows | 11 cols |  2,993 bytes
  • fact_win_distribution.csv    -> 1,008 rows |  5 cols | 23,448 bytes
